# Tree pairwise distance

For each node in a tree, find the distance to each other node, under some assumptions:

* Distance could be simple path length, or we could assume all nodes with a common parent are also only one hop apart

In [1]:
import itertools
import networkx as nx
import numpy as np

In [2]:
g = nx.DiGraph()
node_list = list(range(10))
g.add_nodes_from(node_list)
g.add_edge(0, 1)
g.add_edge(0, 2)
g.add_edge(1, 3)
g.add_edge(1, 4)
g.add_edge(2, 5)
g.add_edge(2, 6)
g.add_edge(2, 7)
g.add_edge(2, 8)
g.add_edge(0, 9)

In [3]:
def shortest_undirected_array(graph):
    sua = np.zeros((len(node_list), len(node_list)), dtype='int')
    for source, target_lengths in nx.all_pairs_shortest_path_length(graph.to_undirected()):
        for target, length in target_lengths.items():
            sua[source, target] = length
    return sua

In [4]:
a = shortest_undirected_array(g)
a

array([[0, 1, 1, 2, 2, 2, 2, 2, 2, 1],
       [1, 0, 2, 1, 1, 3, 3, 3, 3, 2],
       [1, 2, 0, 3, 3, 1, 1, 1, 1, 2],
       [2, 1, 3, 0, 2, 4, 4, 4, 4, 3],
       [2, 1, 3, 2, 0, 4, 4, 4, 4, 3],
       [2, 3, 1, 4, 4, 0, 2, 2, 2, 3],
       [2, 3, 1, 4, 4, 2, 0, 2, 2, 3],
       [2, 3, 1, 4, 4, 2, 2, 0, 2, 3],
       [2, 3, 1, 4, 4, 2, 2, 2, 0, 3],
       [1, 2, 2, 3, 3, 3, 3, 3, 3, 0]])

Now we make children all one hop away at each level

In [5]:
def fully_connect_child_sets(tree, root):
    g2 = tree.copy()
    depths = nx.shortest_path_length(tree, source=root)
    for i in range(max(depths.values())):
        parents = [p for p, d in depths.items() if d == i]
        for p in parents:
            children = list(tree.neighbors(p))
            for c1, c2 in itertools.product(children, repeat=2):
                g2.add_edge(c1, c2)
    return g2

In [6]:
g2 = fully_connect_child_sets(g, node_list[0])

In [7]:
a2 = shortest_undirected_array(g2)
a2

array([[0, 1, 1, 2, 2, 2, 2, 2, 2, 1],
       [1, 0, 1, 1, 1, 2, 2, 2, 2, 1],
       [1, 1, 0, 2, 2, 1, 1, 1, 1, 1],
       [2, 1, 2, 0, 1, 3, 3, 3, 3, 2],
       [2, 1, 2, 1, 0, 3, 3, 3, 3, 2],
       [2, 2, 1, 3, 3, 0, 1, 1, 1, 2],
       [2, 2, 1, 3, 3, 1, 0, 1, 1, 2],
       [2, 2, 1, 3, 3, 1, 1, 0, 1, 2],
       [2, 2, 1, 3, 3, 1, 1, 1, 0, 2],
       [1, 1, 1, 2, 2, 2, 2, 2, 2, 0]])

Which shortens paths between siblings, and children of siblings

In [8]:
a2 - a

array([[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  0, -1,  0,  0, -1, -1, -1, -1, -1],
       [ 0, -1,  0, -1, -1,  0,  0,  0,  0, -1],
       [ 0,  0, -1,  0, -1, -1, -1, -1, -1, -1],
       [ 0,  0, -1, -1,  0, -1, -1, -1, -1, -1],
       [ 0, -1,  0, -1, -1,  0, -1, -1, -1, -1],
       [ 0, -1,  0, -1, -1, -1,  0, -1, -1, -1],
       [ 0, -1,  0, -1, -1, -1, -1,  0, -1, -1],
       [ 0, -1,  0, -1, -1, -1, -1, -1,  0, -1],
       [ 0, -1, -1, -1, -1, -1, -1, -1, -1,  0]])